<a href="https://colab.research.google.com/github/rantawadeesritakorn-tech/Project-Hotel/blob/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88-1/%E0%B9%80%E0%B8%98%C2%84%E0%B9%80%E0%B8%98%C2%99%E0%B9%80%E0%B8%98%E2%80%94%E0%B9%80%E0%B8%98%E0%B8%95%E0%B9%80%E0%B8%99%C2%88_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ส่วนที่ 1 — เตรียมเครื่องมือ


In [ ]:
import math
import os
import random
import sqlite3
from datetime import date, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(7) # กำหนด seed เพื่อให้ผลลัพธ์ซ้ำได้ทุกครั้ง
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
os.makedirs("data", exist_ok=True)
print("พร้อมใช้งาน | pandas", pd.__version__)

พร้อมใช้งาน | pandas 2.2.3


## ส่วนที่ 2 — ค่าคงที่ธุรกิจและโปรไฟล์ตลาด

In [ ]:


# --- ภาษีและค่าบริการ ---
VAT_RATE = 0.07                 # ภาษีมูลค่าเพิ่ม 7%
SERVICE_CHARGE_RATE = 0.10      # ค่าบริการโรงแรม 10%
BREAKFAST_PRICE = 350           # ค่าอาหารเช้า ต่อคืน

# --- ส่วนลดตามระดับสมาชิก ---
MEMBER_DISCOUNT = {"Regular": 0.00, "Silver": 0.05, "Gold": 0.10}
SILVER_THRESHOLD = 3            # พักครบ 3 ครั้ง -> Silver
GOLD_THRESHOLD = 6              # พักครบ 6 ครั้ง -> Gold

# --- ค่าคอมมิชชันตามช่องทางการจอง ---
CHANNEL_COMMISSION = {"Walk-in": 0.00, "Phone": 0.00, "Website": 0.02, "OTA": 0.15}

# --- ฤดูกาลท่องเที่ยว ---
HIGH_SEASON_MONTHS = [1, 2, 12]     # ราคาสูงขึ้น
LOW_SEASON_MONTHS = [5, 6, 9]       # ราคาลดลง

# --- ช่วงเวลาที่จำลอง ---
SIM_START = date(2025, 1, 1)
SIM_END = date(2025, 6, 30)

# --- ผังการอัปเกรดห้องเมื่อห้องที่ขอเต็ม ---
UPGRADE_PATH = {"Standard": "Deluxe", "Deluxe": "Suite", "Family": "Suite"}

# --- ผังห้องพักของโรงแรม (ประเภท, จำนวนห้อง, ราคา/คืน, จำนวนคนพักได้) ---
ROOM_PLAN = [
    ("Standard", 6, 1400, 2),
    ("Deluxe",   5, 2200, 2),
    ("Suite",    2, 4200, 3),
    ("Family",   2, 3000, 4),
]

RANDOM_SEED = 7                 # กลุ่มที่ 7
TARGET_BOOKINGS = 350           # เกณฑ์ขั้นต่ำของโจทย์คือ 300 รายการ

# จำนวนคำขอจองเฉลี่ยต่อวัน (ปรับค่านี้เพื่อคุมอัตราการเข้าพักให้ใกล้ความจริง)
# ตลาดกรุงเทพปี 2025 อัตราการเข้าพักเฉลี่ย ~75%
DAILY_REQUEST_RATE = 14.0
BOOKING_WINDOW_DAYS = 120       # เปิดรับจองล่วงหน้าได้สูงสุดกี่วัน

In [ ]:


# โครงสร้าง: (สัดส่วนตลาด %, จำนวนคืนเฉลี่ย, จองล่วงหน้าเฉลี่ย (วัน),
#             โอกาสซื้ออาหารเช้า, น้ำหนักช่องทาง [OTA, Website, Walk-in, Phone])
MARKET_PROFILE = {
    "Thai": {
        "share": 22, "avg_nights": 1.8, "lead_time": 12, "breakfast": 0.35,
        "channel_weights": [30, 25, 25, 20],
        "first_names": ["สมชาย", "มานี", "วิชัย", "สมหญิง", "ปรีชา", "กมลวรรณ",
                        "ธนากร", "ณัฐพล", "พิมพ์ใจ", "อรวรรณ", "ศิริพร", "จิรายุ"],
        "last_names": ["ใจดี", "รักเรียน", "สายทอง", "ศรีสุข", "พงษ์พันธ์",
                       "วงศ์วิวัฒน์", "แสงจันทร์", "บุญมี", "ทองดี", "อินทรีย์"],
    },
    "Chinese": {
        "share": 18, "avg_nights": 2.6, "lead_time": 25, "breakfast": 0.55,
        "channel_weights": [70, 12, 8, 10],
        "first_names": ["Wei", "Li", "Yan", "Ming", "Hui", "Jing", "Lei", "Xiu"],
        "last_names": ["Zhang", "Wang", "Chen", "Liu", "Yang", "Huang", "Zhao"],
    },
    "Malaysian": {
        "share": 11, "avg_nights": 2.0, "lead_time": 18, "breakfast": 0.45,
        "channel_weights": [60, 18, 12, 10],
        "first_names": ["Ahmad", "Siti", "Tan", "Nurul", "Lim", "Faizal", "Mei Ling"],
        "last_names": ["Abdullah", "Ibrahim", "Wong", "Rahman", "Kumar", "Cheah"],
    },
    "Indian": {
        "share": 8, "avg_nights": 3.2, "lead_time": 30, "breakfast": 0.65,
        "channel_weights": [65, 15, 5, 15],
        "first_names": ["Rahul", "Priya", "Amit", "Neha", "Vikram", "Anjali", "Rajesh"],
        "last_names": ["Sharma", "Patel", "Singh", "Kumar", "Gupta", "Reddy"],
    },
    "Russian": {
        "share": 7, "avg_nights": 5.5, "lead_time": 45, "breakfast": 0.70,
        "channel_weights": [55, 25, 5, 15],
        "first_names": ["Ivan", "Olga", "Dmitry", "Anna", "Sergei", "Ekaterina"],
        "last_names": ["Petrov", "Ivanova", "Smirnov", "Volkov", "Kuznetsova"],
    },
    "Korean": {
        "share": 6, "avg_nights": 2.8, "lead_time": 28, "breakfast": 0.50,
        "channel_weights": [68, 17, 5, 10],
        "first_names": ["Min-jun", "Ji-woo", "Seo-yeon", "Do-hyun", "Ha-eun"],
        "last_names": ["Kim", "Lee", "Park", "Choi", "Jung", "Kang"],
    },
    "Japanese": {
        "share": 5, "avg_nights": 3.0, "lead_time": 35, "breakfast": 0.60,
        "channel_weights": [50, 25, 5, 20],
        "first_names": ["Haruto", "Yui", "Sota", "Sakura", "Ren", "Aoi"],
        "last_names": ["Sato", "Suzuki", "Takahashi", "Tanaka", "Watanabe"],
    },
    "British": {
        "share": 5, "avg_nights": 4.5, "lead_time": 48, "breakfast": 0.65,
        "channel_weights": [55, 28, 5, 12],
        "first_names": ["James", "Emma", "Oliver", "Sophie", "Harry", "Charlotte"],
        "last_names": ["Smith", "Jones", "Taylor", "Brown", "Wilson", "Davies"],
    },
    "German": {
        "share": 5, "avg_nights": 5.0, "lead_time": 50, "breakfast": 0.70,
        "channel_weights": [52, 30, 4, 14],
        "first_names": ["Lukas", "Anna", "Felix", "Lena", "Jonas", "Marie"],
        "last_names": ["Müller", "Schmidt", "Schneider", "Fischer", "Weber"],
    },
    "American": {
        "share": 5, "avg_nights": 4.0, "lead_time": 42, "breakfast": 0.60,
        "channel_weights": [55, 27, 6, 12],
        "first_names": ["Michael", "Jessica", "David", "Ashley", "Chris", "Sarah"],
        "last_names": ["Johnson", "Williams", "Miller", "Davis", "Garcia"],
    },
    "Singaporean": {
        "share": 4, "avg_nights": 2.2, "lead_time": 20, "breakfast": 0.50,
        "channel_weights": [62, 20, 8, 10],
        "first_names": ["Wei Ming", "Jia Hui", "Ryan", "Shermaine", "Daniel"],
        "last_names": ["Tan", "Lim", "Ng", "Koh", "Goh", "Teo"],
    },
    "Australian": {
        "share": 4, "avg_nights": 4.2, "lead_time": 40, "breakfast": 0.60,
        "channel_weights": [58, 26, 6, 10],
        "first_names": ["Jack", "Chloe", "Liam", "Zoe", "Ethan", "Mia"],
        "last_names": ["Anderson", "Thompson", "White", "Harris", "Martin"],
    },
}

NATIONALITIES = list(MARKET_PROFILE.keys())
NATIONALITY_WEIGHTS = [MARKET_PROFILE[n]["share"] for n in NATIONALITIES]

# ความชอบประเภทห้องต่างกันตามกลุ่มผู้เข้าพัก
# (เดินทางคนเดียว / คู่รัก / ครอบครัว / กลุ่มเพื่อน)
TRAVEL_PARTY = {
    "solo":   {"weight": 22, "adults": 1, "children": 0,
               "room_weights": {"Standard": 70, "Deluxe": 26, "Suite": 3, "Family": 1}},
    "couple": {"weight": 48, "adults": 2, "children": 0,
               "room_weights": {"Standard": 38, "Deluxe": 45, "Suite": 15, "Family": 2}},
    "family": {"weight": 22, "adults": 2, "children": 2,
               "room_weights": {"Standard": 8, "Deluxe": 20, "Suite": 17, "Family": 55}},
    "group":  {"weight": 8, "adults": 3, "children": 0,
               "room_weights": {"Standard": 30, "Deluxe": 30, "Suite": 15, "Family": 25}},
}

# อัตราการยกเลิกตามช่องทาง (Cloudbeds 2026 report: OTA 21.8% / direct 10.6%)
CANCEL_RATE_BY_CHANNEL = {
    "OTA": 0.218,
    "Website": 0.106,
    "Phone": 0.100,
    "Walk-in": 0.010,       # จ่ายเงินหน้าเคาน์เตอร์แล้ว แทบไม่ยกเลิก
}

NO_SHOW_RATE = 0.025        # ไม่มาเช็คอินโดยไม่แจ้ง ~2-3% ของใบจองที่ไม่ยกเลิก

# ตัวคูณจำนวนคนเข้าพักตามวันในสัปดาห์
# (สถิติจริง: เช็คอินวันศุกร์ 18.5% วันเสาร์ 17.2% สูงสุดของสัปดาห์)
WEEKDAY_DEMAND = {
    0: 0.80,   # จันทร์
    1: 0.80,   # อังคาร
    2: 0.85,   # พุธ
    3: 1.00,   # พฤหัสบดี
    4: 1.55,   # ศุกร์
    5: 1.45,   # เสาร์
    6: 0.95,   # อาทิตย์
}

# ตัวคูณดีมานด์รายเดือน (High season ต้นปีคนเยอะ / พ.ค.-มิ.ย. เป็น low season)
MONTH_DEMAND = {
    1: 1.25, 2: 1.20, 3: 1.05, 4: 1.00, 5: 0.85, 6: 0.80,
    7: 0.90, 8: 0.95, 9: 0.80, 10: 1.00, 11: 1.15, 12: 1.30,
}

## ส่วนที่ 3 — ออกแบบ Class : Room และ Guest

In [ ]:

class Room:
    """ห้องพัก 1 ห้อง - จำ 'วันที่ถูกจองไปแล้ว' ของตัวเองไว้ในตัวเอง"""

    def __init__(self, room_id, room_number, room_type, base_price, capacity, floor):
        self.room_id = room_id
        self.room_number = room_number
        self.room_type = room_type          # Standard / Deluxe / Suite / Family
        self.base_price = base_price        # ราคาต่อคืนก่อนบวกฤดูกาล
        self.capacity = capacity
        self.floor = floor
        self.booked_dates = set()           # เก็บวันที่ถูกจองแล้ว

    def is_available(self, check_in, nights):
        """ตรวจสอบว่าห้องนี้ว่างตลอดช่วง check_in ถึง check_in+nights หรือไม่"""
        for i in range(nights):
            if (check_in + timedelta(days=i)) in self.booked_dates:
                return False                # เจอวันชนกันแม้แต่วันเดียว = จองไม่ได้
        return True

    def reserve(self, check_in, nights):
        """ยึดวันทั้งช่วงไว้ (เรียกหลังยืนยันการจองสำเร็จแล้วเท่านั้น)"""
        for i in range(nights):
            self.booked_dates.add(check_in + timedelta(days=i))

    def release(self, check_in, nights):
        """คืนวันว่างกลับเข้าระบบ - ใช้ตอนลูกค้ายกเลิกการจอง"""
        for i in range(nights):
            self.booked_dates.discard(check_in + timedelta(days=i))

    def price_per_night(self, check_in):
        """ราคาต่อคืนจริง = ราคาฐาน x ตัวคูณฤดูกาล"""
        return round(self.base_price * seasonal_multiplier(check_in), 2)

    def nights_sold(self):
        """จำนวนคืนที่ห้องนี้ขายได้ทั้งหมด"""
        return len(self.booked_dates)

    def __repr__(self):
        return f"Room({self.room_number}, {self.room_type}, {self.base_price} THB)"


def build_rooms():
    """สร้างห้องพักทั้งหมดตามผังใน config -- return list ของ Room object"""
    rooms, room_id, number = [], 1, 101
    for room_type, count, price, capacity in ROOM_PLAN:
        for _ in range(count):
            floor = number // 100
            rooms.append(Room(room_id, str(number), room_type, price, capacity, floor))
            room_id += 1
            number += 1
            if number % 100 > 10:           # ขึ้นชั้นใหม่ทุก 10 ห้อง
                number = (number // 100 + 1) * 100 + 1
    return rooms


# %% [code]
class Guest:
    """ลูกค้า 1 คน - ลูกค้าคนเดิมจองได้หลายใบ จึงต้องแยกเป็นอีกตาราง"""

    def __init__(self, guest_id, name, phone, nationality, member_tier="Regular"):
        self.guest_id = guest_id
        self.name = name
        self.phone = phone
        self.nationality = nationality
        self.member_tier = member_tier      # Regular / Silver / Gold
        self.stay_count = 0

    def discount_rate(self):
        """คืนอัตราส่วนลดตามระดับสมาชิก"""
        return MEMBER_DISCOUNT[self.member_tier]

    def upgrade_tier(self):
        """เงื่อนไขพิเศษ: พักครบตามเกณฑ์แล้วเลื่อนขั้นสมาชิกอัตโนมัติ"""
        if self.stay_count >= GOLD_THRESHOLD:
            self.member_tier = "Gold"
        elif self.stay_count >= SILVER_THRESHOLD:
            self.member_tier = "Silver"

    def record_stay(self):
        """บันทึกว่าลูกค้าคนนี้พักเพิ่มอีก 1 ครั้ง แล้วเช็คเลื่อนระดับ"""
        self.stay_count += 1
        self.upgrade_tier()

    def is_vip(self):
        """ลูกค้า VIP = ระดับ Gold ขึ้นไป"""
        return self.member_tier == "Gold"

    def __repr__(self):
        return f"Guest({self.guest_id}, {self.name}, {self.member_tier})"